*— cell 0 —*

# Arm-2C — Agentic tree navigation (Azure GPU run)

A ReAct agent (LLaMA 3.1 8B, vectorless) descends a **deep** AzureDI tree
(LIVRE › TITRE › CHAPITRE › Section › Article) rebuilt from the full header
stack — testing whether a better-built tree fixes Arm-2B/PageIndex's
wrong-chapter miss.

**One run = one PDF × one mode.** Sequence: GPU → Ollama → clone repo →
pip install ollama → load tree+queries → warmup → single-query smoke →
5-query pilot + ETA gate → full run → **success/failure check**.

Resume-friendly: any `q<qid>.json` already on disk is skipped.


In [10]:
# === cell 1 ===
# ── PER-RUN CONFIG (edit these) ─────────────────────────────────────────────
DOC_ID    = "1804_03_21_1804032150"   # best-chance PDF: Code Civil — most gold (252 q),
                                       # richest deep tree (depth 7), biggest flat->deep contrast
MODE      = "enriched"                 # "enriched" = FR labels + native-EN level_summary (branches)
                                       # + content_summary/keywords (articles); "bare" = labels only.
                                       # bare baseline already run (R@10 0.166); enriched is the test.
MAX_NODES = 40                         # frontier-descent budget: nodes visited per query (= LLM calls)
MAX_BRANCH = 5                         # max sub-sections descended per node (caps frontier/cost; T05 used 5)
RERANK    = True                       # +1 call/query: re-rank the navigated pool by relevance.
                                       # enriched (no rerank) gave R@10 0.214 with R@100 0.515 -> rerank
                                       # converts that reachable gold into the top-10.

# ── REPO + AUTH ─────────────────────────────────────────────────────────────
GITHUB_TOKEN  = ""                     # PAT with read scope (leave blank if public)
GITHUB_OWNER  = "MariusPasch"
MONO_REPO     = "bsard-rag-thesis"
GITHUB_REPO   = "RQ2_T04_ARM2_METADATA"   # now a subfolder under RQ2_Structure_Aware_Retrieval/
GITHUB_BRANCH = "main"

# ── VM PATHS ────────────────────────────────────────────────────────────────
REPOS_DIR    = "/home/azureuser/repos"
RESULTS_ROOT = "/home/azureuser/results"
OLLAMA_MODEL = "llama3.1:8b"

RUN_NAME = f"arm2c_{DOC_ID}_{MODE}" + ("_rerank" if RERANK else "")
print("RUN_NAME   ", RUN_NAME)
print(f"DOC_ID {DOC_ID} | MODE {MODE} | MAX_NODES {MAX_NODES} | RERANK {RERANK}")


RUN_NAME    arm2c_1804_03_21_1804032150_enriched_rerank
DOC_ID 1804_03_21_1804032150 | MODE enriched | MAX_NODES 40 | RERANK True


*— cell 2 —*

## 1. GPU sanity check

In [11]:
# === cell 3 ===
!nvidia-smi


Fri Jun  5 13:55:19 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.274.02             Driver Version: 535.274.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       On  | 00000001:00:00.0 Off |                  Off |
| N/A   44C    P0              28W /  70W |   6953MiB / 16384MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

*— cell 4 —*

## 2. Ollama install / start / pull

Idempotent: skips install if present, skips start if 11434 serves, skips
pull if cached. KV-cache q8_0 keeps the 16k context inside a T4's VRAM.

In [12]:
# === cell 5 ===
import os, subprocess, time, urllib.request

if subprocess.run(["which", "ollama"], capture_output=True, text=True).returncode != 0:
    print("Installing Ollama...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

os.environ["OLLAMA_NUM_GPU"] = "99"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q8_0"
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"

def _up(url="http://localhost:11434/api/tags"):
    try:
        with urllib.request.urlopen(url, timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if not _up():
    print("Starting ollama serve...")
    subprocess.Popen(["bash", "-c", "ollama serve > /tmp/ollama.log 2>&1 &"])
    for _ in range(30):
        if _up():
            break
        time.sleep(1)
    else:
        raise RuntimeError("ollama serve not reachable on 11434 in 30s. See /tmp/ollama.log.")

print("Pulling model (no-op if cached)...")
subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)
print("Ollama ready.")


Pulling model (no-op if cached)...


pulling manifest ⠋ pulling manifest ⠹ pulling manifest ⠹ 

Ollama ready.


pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


*— cell 6 —*

## 3. Clone the T04 repo (carries Arm-2C code + tree + query bundle)

In [13]:
# === cell 7 ===
import subprocess
from pathlib import Path

# Arm-2C lives at RQ2_Structure_Aware_Retrieval/RQ2_T04_ARM2_METADATA/experiments/arm2c
# inside the single mono-repo. Clone the mono-repo once, then point at the subfolder.
parent = Path(REPOS_DIR); parent.mkdir(parents=True, exist_ok=True)
MONO_DIR = parent / MONO_REPO
auth = (f"https://{GITHUB_TOKEN}@github.com/{GITHUB_OWNER}/{MONO_REPO}.git"
        if GITHUB_TOKEN else f"https://github.com/{GITHUB_OWNER}/{MONO_REPO}.git")

if MONO_DIR.exists():
    subprocess.run(["git", "-C", str(MONO_DIR), "remote", "set-url", "origin", auth], check=True)
    subprocess.run(["git", "-C", str(MONO_DIR), "fetch", "--quiet"], check=True)
    subprocess.run(["git", "-C", str(MONO_DIR), "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", str(MONO_DIR), "pull", "--quiet"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", GITHUB_BRANCH, auth, str(MONO_DIR)], check=True)

# The former per-task repo is now a subfolder of the mono-repo.
RQ2_ROOT = MONO_DIR / "RQ2_Structure_Aware_Retrieval"
T04_DIR = RQ2_ROOT / GITHUB_REPO

head = subprocess.run(["git", "-C", str(MONO_DIR), "log", "-1", "--oneline"],
                      capture_output=True, text=True).stdout.strip()
print("repo:", MONO_DIR, "|", head)
print("arm2c subfolder:", T04_DIR)


Already on 'master'


Your branch is up to date with 'origin/master'.
repo: /home/azureuser/repos/RQ2_T04_ARM2_METADATA | 5a84312 enrichment and reranking


*— cell 8 —*

## 4. Install `ollama` + add Arm-2C to the path

In [14]:
# === cell 9 ===
import subprocess, sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ollama"], check=True)
EXP_DIR = str(T04_DIR / "experiments" / "arm2c")
if EXP_DIR not in sys.path:
    sys.path.insert(0, EXP_DIR)

import navigator_tools, react_navigator, check_results   # stdlib + ollama only
print("Arm-2C modules import OK from", EXP_DIR)


Arm-2C modules import OK from /home/azureuser/repos/RQ2_T04_ARM2_METADATA/experiments/arm2c


*— cell 10 —*

## 5. Load deep tree, queries, baselines

In [15]:
# === cell 11 ===
import json
from pathlib import Path
from navigator_tools import DeepTree

EXP = T04_DIR / "experiments" / "arm2c"
bundle = EXP / "bundles" / DOC_ID
tree = DeepTree.load(bundle / "deep_tree.json")   # tree shipped inside the bundle
queries = json.loads((bundle / "queries.json").read_text(encoding="utf-8"))
baselines = json.loads((bundle / "baselines.json").read_text(encoding="utf-8"))

print(f"tree nodes : {len(tree.by_id)}  (depth {tree.manifest.get('stat_max_depth')}, "
      f"orphan {tree.manifest.get('stat_orphan_pct')}%)")
print(f"queries    : {len(queries)}")
print(f"baselines recall@10 : {baselines['baselines_recall@10']}")
print(f"  -> bar to clear (Arm-2B): {baselines['baselines_recall@10'].get('arm2b')}")


tree nodes : 568  (depth 7, orphan 6.5%)
queries    : 252
baselines recall@10 : {'arm1': {'mean_recall@10': 0.4713, 'n_questions': 252}, 'arm2a': {'mean_recall@10': 0.5137, 'n_questions': 252}, 'arm2b': {'mean_recall@10': 0.2026, 'n_questions': 252}}
  -> bar to clear (Arm-2B): {'mean_recall@10': 0.2026, 'n_questions': 252}


*— cell 12 —*

## 6. Pre-flight smokes\n\n### 6a. LLM warmup (3 cold calls)

In [16]:
# === cell 13 ===
import time
from react_navigator import LlamaClient

llm = LlamaClient(model=OLLAMA_MODEL, num_ctx=16384)
for i in range(3):
    t0 = time.perf_counter()
    txt = llm.generate('Reponds uniquement en JSON: {"ok": true}')
    print(f"call {i+1}: {(time.perf_counter()-t0)*1000:7.0f} ms   text={txt[:60]!r}")


call 1:     410 ms   text='{"ok": true}'
call 2:     308 ms   text='{"ok": true}'
call 3:     318 ms   text='{"ok": true}'


*— cell 14 —*

### 6b. Single-query end-to-end (confirms FR-JSON descent + trace)

In [17]:
# === cell 15 ===
import time
from react_navigator import navigate

q = queries[0]
print(f"qid={q['query_id']}  {q['query_text'][:90]!r}\n  gold={q['gold_bsard_ids'][:12]}\n")
t0 = time.perf_counter()
res = navigate(q["query_text"], tree, llm, mode=MODE, max_nodes=MAX_NODES, max_branch=MAX_BRANCH, rerank=RERANK)
print(f"{time.perf_counter()-t0:.1f}s  visited={res.nodes_visited}  calls={res.llm_calls}  "
      f"exit={res.exit_reason}")
print(f"selected bsard_ids: {res.selected_bsard_ids[:12]}")
print("steps:")
for s in res.steps:
    flag = "" if s.parse_ok else "  PARSE_FAIL"
    print(f"  [{s.title[:38]:38}] sec={len(s.sections)} art={len(s.articles)}  "
          f"{s.latency_ms:6.0f}ms{flag}")


qid=1  'Les régimes de protection des personnes fragilisées ?'
  gold=[1355, 1356, 1357, 1358, 1359, 1360, 1361, 1362, 1363, 1364, 1365, 1366]

43.0s  visited=12  calls=13  exit=frontier_empty
selected bsard_ids: [1323, 1324, 1326, 1327, 1328]
steps:
  [21 MARS 1804 - CODE CIVIL             ] sec=1 art=0    2281ms
  [LIVRE I. - DES PERSONNES.             ] sec=2 art=0    3249ms
  [TITRE IX. [1 - De l'autorité parentale] sec=2 art=0    3042ms
  [CHAPITRE Ier. [1 - De l'autorité paren] sec=0 art=0    3470ms
  [CHAPITRE II. [1 - De l'accueil familia] sec=0 art=0    3428ms
  [TITRE X. - DE LA MINORITE.DE LA TUTELL] sec=2 art=0    2821ms
  [CHAPITRE II. < L 2001-04-29/39, art. 1] sec=4 art=0    2778ms
  [Section II. < L 2001-04-29/39, art. 13] sec=0 art=0    2566ms
  [Section III. < L 2001-04-29/39, art. 1] sec=0 art=0    3058ms
  [Section IV. < L 2001-04-29/39, art. 13] sec=0 art=5    3031ms
  [Section V. < L 2001-04-29/39, art. 13,] sec=0 art=0    3012ms
  [CHAPITRE IIbis. - DE LA TUTELLE

*— cell 16 —*

### 6c. 5-query pilot + time-budget gate

**Continue past this cell only if the projected total and parse-fail rate
look acceptable.** The next cell starts the full run.

In [18]:
# === cell 17 ===
import statistics, time
from react_navigator import navigate

pilot = queries[: min(5, len(queries))]
calls, lat, pf, tot = [], [], 0, 0
for q in pilot:
    t0 = time.perf_counter()
    r = navigate(q["query_text"], tree, llm, mode=MODE, max_nodes=MAX_NODES, max_branch=MAX_BRANCH, rerank=RERANK)
    lat.append(time.perf_counter() - t0)
    calls.append(r.llm_calls)
    pf += sum(1 for s in r.steps if not s.parse_ok)
    tot += r.llm_calls

mean_lat = statistics.mean(lat)
print(f"pilot ({len(pilot)} q): calls/q mean={statistics.mean(calls):.1f}  "
      f"s/q mean={mean_lat:.1f}  parse-fail={pf}/{tot} ({100*pf/max(tot,1):.1f}%)")
print(f"PROJECTED full run on {len(queries)} q: ~{mean_lat*len(queries)/60:.0f} min")
print("Inspect, then run the full-run cell.")


pilot (5 q): calls/q mean=11.0  s/q mean=39.1  parse-fail=0/55 (0.0%)
PROJECTED full run on 252 q: ~164 min
Inspect, then run the full-run cell.


*— cell 18 —*

### Fresh-run guard

The full run is resume-friendly (skips any `q<qid>.json` already on disk) — great
for resuming an interrupted run, but it means a **code change silently reuses old
results**. Set `FRESH_RUN = True` to wipe THIS `RUN_NAME`'s results for a clean
from-scratch run; `False` to resume.

In [19]:
# === cell 19 ===
import shutil
from pathlib import Path

FRESH_RUN = True   # True = wipe and start clean; False = resume an interrupted run
#FRESH_RUN = False 

RESULTS = Path(RESULTS_ROOT) / RUN_NAME
if FRESH_RUN and RESULTS.exists():
    shutil.rmtree(RESULTS)
    print("wiped", RESULTS)
RESULTS.mkdir(parents=True, exist_ok=True)
print(f"FRESH_RUN={FRESH_RUN}  results dir: {RESULTS}")


wiped /home/azureuser/results/arm2c_1804_03_21_1804032150_enriched_rerank
FRESH_RUN=True  results dir: /home/azureuser/results/arm2c_1804_03_21_1804032150_enriched_rerank


*— cell 20 —*

## 7. Full run (resume-friendly — skips any q<qid>.json already on disk)

In [20]:
# === cell 21 ===
import json, time
from dataclasses import asdict
from pathlib import Path
from react_navigator import navigate

RESULTS = Path(RESULTS_ROOT) / RUN_NAME
RESULTS.mkdir(parents=True, exist_ok=True)
done = {p.stem for p in RESULTS.glob("q*.json")}
print(f"resuming: {len(done)}/{len(queries)} already done")

t0 = time.perf_counter()
for i, q in enumerate(queries, 1):
    qid = str(q["query_id"])
    if f"q{qid}" in done:
        continue
    r = navigate(q["query_text"], tree, llm, mode=MODE, max_nodes=MAX_NODES, max_branch=MAX_BRANCH, rerank=RERANK)
    rec = {
        "query_id": qid, "query_text": q["query_text"], "gold_bsard_ids": q["gold_bsard_ids"],
        "selected_bsard_ids": r.selected_bsard_ids, "ranked_bsard_ids": r.ranked_bsard_ids,
        "ranked_bsard_ids_prererank": r.ranked_bsard_ids_prererank,
        "nodes_visited": r.nodes_visited, "llm_calls": r.llm_calls,
        "exit_reason": r.exit_reason, "mode": MODE,
        "steps": [asdict(s) for s in r.steps],
    }
    (RESULTS / f"q{qid}.json").write_text(json.dumps(rec, ensure_ascii=False), encoding="utf-8")
    if i % 10 == 0:
        print(f"  {i}/{len(queries)}  ({(time.perf_counter()-t0)/max(i-len(done),1):.1f}s/q)")

print(f"\nDONE — {len(list(RESULTS.glob('q*.json')))} results in {RESULTS}")


resuming: 0/252 already done
  10/252  (36.5s/q)
  20/252  (40.6s/q)
  30/252  (36.1s/q)
  40/252  (36.5s/q)
  50/252  (36.3s/q)
  60/252  (37.6s/q)


: 

: 

*— cell 22 —*

## 8. Analysis & decision

Not just pass/fail — a full report to decide **scale to the other PDFs, or change
the approach (and what)**: headline vs baselines, navigation behaviour, and the key
**miss decomposition** — every gold article classified as HIT / SEEN_NOT_SELECTED
(seen in a menu, not picked → selection problem) / NOT_REACHED (branch pruned → tree
problem, the Arm-2B failure mode) / ORPHAN_UNREACHED (Unfiled). Plus per-cardinality
recall, worst/best queries, and a rule-based recommendation. Writes
`analysis_report.md` + `per_query.csv` next to the results.

In [ ]:
# === cell 23 ===
import importlib, analyze_results
importlib.reload(analyze_results)
summary = analyze_results.analyze(RESULTS, bundle, bundle / "deep_tree.json")


ARM-2C ANALYSIS — arm2c_1804_03_21_1804032150_enriched_rerank  (252/252 queries scored)

[1] HEADLINE — recall@k / hit / MRR
  R@1=0.026  R@5=0.065  R@10=0.091  R@20=0.107  R@100=0.218
  hit@10=0.127   MRR@10=0.087
  baselines recall@10:  Arm-1=0.4713  Arm-2A=0.5137  Arm-2B=0.2026
  Arm-2C vs Arm-2B: -0.111 (-55% rel)  BELOW the bar

[1b] RERANK EFFECT (same navigation, ranking before vs after the re-rank call)
  recall@10 : 0.070 (pre) -> 0.091 (post)   delta +0.022
  recall@100: 0.218 (pre) -> 0.218 (post)  (rerank reorders the pool; R@100 ~unchanged)

[2] NAVIGATION
  nodes visited/q: mean=3.7 median=1 max=22
  exit reasons: {'frontier_empty': 252}   (budget-hits => raise MAX_NODES)
  LLM calls=931  parse-fail=187 (20.1%)
  queries that explored the Unfiled branch: 2/252

[3] SELECTION
  selected/q: mean=0.6 median=0  empty-selection queries=212
  precision (selected that are gold): 0.016

[4] MISS DECOMPOSITION  (the key diagnostic — total gold = 973, misses = 959)
  HIT           

*— cell 24 —*

## 9. Copy results down + cleanup

Per-query `q<qid>.json` + `analysis_report.md` + `per_query.csv` are under
`RESULTS_ROOT/RUN_NAME/`. Pull them into the **committable** `runs/` dir (NOT
`results/`, which is gitignored) so they persist for future analysis /
bare-vs-enriched compare:

```
scp -r azureuser@<vm-ip>:/home/azureuser/results/<RUN_NAME> \
       "<repo>/RQ2_T04_ARM2_METADATA/experiments/arm2c/runs/"
```

In [ ]:
# === cell 25 ===
import subprocess
subprocess.run(["pkill", "-f", "ollama"], check=False)
print("Ollama stopped. Results in:", RESULTS)


Ollama stopped. Results in: /home/azureuser/results/arm2c_1804_03_21_1804032150_enriched_rerank


pkill: killing pid 11261 failed: Operation not permitted
